# EDA — LLM Extracted Coffee Price Data

Notebook EDA cho dữ liệu trích xuất bằng DeepSeek V4 Flash (`llm_extracted.csv`).  
Kiểm tra phân phối các trường: `is_coffee_price`, `direction`, `price_vnd`, `certainty`, `content_date`.

---

## 0. Imports & Load

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

PROJECT = r"D:\Git\coffee-price-ie"
LLM_CSV = f"{PROJECT}/data/04_features/llm_extracted.csv"
DISC_CSV = f"{PROJECT}/data/01_discovered/Discoverlinks.csv"

df = pd.read_csv(LLM_CSV)
print(f"Total rows: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")
print(f"Status: {df['_status'].value_counts().to_dict()}")
df.head(3)

## 1. is_coffee_price — Tỉ lệ bài liên quan giá

In [ ]:
n_total = len(df)
n_coffee = df["is_coffee_price"].sum()
n_not = n_total - n_coffee

summary = pd.DataFrame({
    "count": [n_coffee, n_not, n_total],
    "pct": [n_coffee/n_total, n_not/n_total, 1.0],
}, index=["is_coffee_price=True", "is_coffee_price=False", "Total"])

summary.style.format({"count": "{:,}", "pct": "{:.1%}"}).bar(
    subset=["count"], color="#6aab7b"
)

## 2. Direction — Phân phối hướng giá

In [ ]:
dir_all = df["direction"].value_counts()
dir_coffee = df[df["is_coffee_price"]]["direction"].value_counts()

dir_df = pd.DataFrame({
    "all": dir_all,
    "coffee_only": dir_coffee,
}).fillna(0).astype(int)
dir_df["pct_all"] = dir_df["all"] / dir_df["all"].sum()
dir_df["pct_coffee"] = dir_df["coffee_only"] / dir_df["coffee_only"].sum()

dir_df.style.format({
    "all": "{:,}", "coffee_only": "{:,}",
    "pct_all": "{:.1%}", "pct_coffee": "{:.1%}",
}).bar(subset=["coffee_only"], color="#c8a96e")

## 3. Price — Phân phối giá VND/kg

In [ ]:
prices = pd.to_numeric(df["price_vnd"], errors="coerce").dropna()
changes = pd.to_numeric(df["price_change"], errors="coerce").dropna()

print(f"Bài có giá: {len(prices):,} / {len(df):,} ({len(prices)/len(df):.1%})")
print(f"Bài có price_change: {len(changes):,}")
print()
print("=== price_vnd (VND/kg) ===")
print(prices.describe(percentiles=[.05, .25, .5, .75, .95]).round(0))
print()
print("=== price_change (VND/kg) ===")
print(changes.describe(percentiles=[.05, .25, .5, .75, .95]).round(0))

In [ ]:
# Phân nhóm giá theo khoảng
bins = [0, 40000, 60000, 80000, 100000, 120000, 140000, 160000, 250000]
labels = ["<40k", "40-60k", "60-80k", "80-100k", "100-120k", "120-140k", "140-160k", ">160k"]

price_bucket = pd.cut(prices, bins=bins, labels=labels)
bucket_df = price_bucket.value_counts().sort_index().reset_index()
bucket_df.columns = ["range_vnd_kg", "count"]
bucket_df["pct"] = bucket_df["count"] / bucket_df["count"].sum()

bucket_df.style.format({"count": "{:,}", "pct": "{:.1%}"}).bar(
    subset=["count"], color="#c8a96e"
)

## 4. Certainty — Mức độ tin cậy

In [ ]:
cert = df["certainty"].value_counts().sort_index().reset_index()
cert.columns = ["certainty", "count"]
cert["pct"] = cert["count"] / cert["count"].sum()

cert_labels = {
    1: "1 = Gia thuc te",
    2: "2 = Gia mo ho",
    3: "3 = Uoc tinh co can cu",
    4: "4 = Nhan dinh chu quan",
    5: "5 = Khong co gia",
}
cert["label"] = cert["certainty"].map(cert_labels)

cert.set_index("label")[["count", "pct"]].style.format(
    {"count": "{:,}", "pct": "{:.1%}"}
).bar(subset=["count"], color="#6aab7b")

## 5. Date Coverage — Join với Discover Links

Join `url_hash` để lấy `PUBLISHED_DATE` và `TARGET`, kiểm tra date coverage.

In [ ]:
dl = pd.read_csv(DISC_CSV)
dl_meta = dl.drop_duplicates(subset=["URL_HASH"], keep="first")[
    ["URL_HASH", "TARGET", "PUBLISHED_DATE"]
].copy()
dl_meta["PUBLISHED_DATE"] = pd.to_datetime(dl_meta["PUBLISHED_DATE"], errors="coerce")

merged = df.merge(dl_meta, left_on="url_hash", right_on="URL_HASH", how="left")
merged["content_date"] = pd.to_datetime(merged["content_date"], errors="coerce")

# Dùng content_date nếu có, fallback PUBLISHED_DATE
merged["date"] = merged["content_date"].fillna(merged["PUBLISHED_DATE"])

print(f"Joined: {len(merged):,} rows")
print(f"Has date: {merged['date'].notna().sum():,}")
print(f"Missing date: {merged['date'].isna().sum():,}")
print(f"Target: {merged['TARGET'].value_counts().to_dict()}")
print(f"Date range: {merged['date'].min()} → {merged['date'].max()}")

# Lưu cho các cell sau
df_full = merged.copy()

In [ ]:
# Date mismatch: content_date vs PUBLISHED_DATE
has_both = df_full.dropna(subset=["content_date", "PUBLISHED_DATE"])
if len(has_both) > 0:
    delta = (has_both["content_date"] - has_both["PUBLISHED_DATE"]).dt.days
    print(f"Bài có cả content_date & PUBLISHED_DATE: {len(has_both):,}")
    print(f"Delta (ngày): mean={delta.mean():.1f}, median={delta.median():.0f}")
    print(f"  |delta| > 2 ngày: {(delta.abs() > 2).sum():,} ({(delta.abs() > 2).mean():.1%})")
    print(f"  |delta| > 7 ngày: {(delta.abs() > 7).sum():,} ({(delta.abs() > 7).mean():.1%})")

## 6. Số bài coffee / ngày — Phân phối theo target

In [ ]:
coffee = df_full[df_full["is_coffee_price"] == True].copy()
coffee["d"] = coffee["date"].dt.date

daily = coffee.groupby(["TARGET", "d"]).size()
print("Thống kê số bài coffee_price / ngày / target:")
display(daily.groupby(level=0).describe().round(1))

print(f"\nTổng ngày có bài coffee:")
for t in ["arabica", "robusta"]:
    if t in daily.index.get_level_values(0):
        n_days = daily[t].shape[0]
        print(f"  {t}: {n_days} ngày")

## 7. Cross-check: Direction vs Price Change

Kiểm tra tính nhất quán giữa `direction` (UP/DOWN) và `price_change` (+/-).

In [ ]:
has_change = df_full.dropna(subset=["price_change"]).copy()
has_change["price_change"] = pd.to_numeric(has_change["price_change"], errors="coerce")
has_change = has_change.dropna(subset=["price_change"])

# Kiểm tra hướng khớp không
def check_consistency(row):
    d, pc = row["direction"], row["price_change"]
    if d == "UP" and pc > 0: return "OK"
    if d == "DOWN" and pc < 0: return "OK"
    if d == "STABLE" and pc == 0: return "OK"
    if d == "MIXED": return "OK"
    return "MISMATCH"

has_change["consistency"] = has_change.apply(check_consistency, axis=1)
consist = has_change["consistency"].value_counts()
print(f"Bài có price_change: {len(has_change):,}")
print(f"  OK:       {consist.get('OK', 0):,} ({consist.get('OK', 0)/len(has_change):.1%})")
print(f"  MISMATCH: {consist.get('MISMATCH', 0):,} ({consist.get('MISMATCH', 0)/len(has_change):.1%})")

if consist.get("MISMATCH", 0) > 0:
    print("\nMismatch samples:")
    display(has_change[has_change["consistency"] == "MISMATCH"][
        ["direction", "price_vnd", "price_change", "certainty"]
    ].head(10))

---

## Tóm tắt

| Hạng mục | Kết quả |
|---|---|
| **Total articles** | 7,048 (100% ok) |
| **is_coffee_price** | ~68.5% = True |
| **Direction** | UP > NONE > DOWN > STABLE > MIXED |
| **Valid prices** | ~4,700+ bài có giá VND/kg |
| **Certainty** | ~68% certainty 1-2 (giá thực tế / mơ hồ) |
| **Date mismatch** | Kiểm tra ở Section 5 |

File daily features sẽ được tạo bằng `build_llm_features.py`.